# LeetCode #97 — Interleaving String: DP trên lưới hai chuỗi

**Mức độ:** Medium · **Chủ đề:** String, Dynamic Programming

Skeleton trên LeetCode:

```python
class Solution:
    def isInterleave(self, s1: str, s2: str, s3: str) -> bool:
```

Mục tiêu của notebook:

1. Thấy vì sao **tham lam** (khớp `s1` trước nếu được) sai.
2. Trạng thái `dp[i][j]`: `s1[:i]` và `s2[:j]` có đan thành `s3[:i+j]` không.
3. Hình dung bài toán như **tìm đường trên lưới** (chỉ đi phải / xuống), giống #62.
4. Tối ưu bộ nhớ một hàng (follow-up của đề).

# 1. Bài toán là gì?

Cho ba chuỗi `s1`, `s2`, `s3`. Kiểm tra `s3` có phải là **đan xen (interleaving)** của `s1` và `s2` không.

Đan xen: chia `s1` thành các đoạn `s1 = a1 + a2 + …`, `s2 = b1 + b2 + …`, rồi ghép xen kẽ `a1 + b1 + a2 + b2 + …` (hoặc bắt đầu bằng `b1`). Nói cách khác: `s3` được tạo bằng cách **trộn** các ký tự của `s1` và `s2`, **giữ nguyên thứ tự** bên trong mỗi chuỗi.

**Constraints (theo đề):**

- `0 <= s1.length, s2.length <= 100`
- `0 <= s3.length <= 200`
- `s1`, `s2`, `s3` gồm chữ cái tiếng Anh viết thường.

**Follow up (theo đề):** giải được với bộ nhớ phụ chỉ `O(s2.length)` không?

---

## Ví dụ

```text
s1 = "aabcc", s2 = "dbbca", s3 = "aadbbcbcac"  ->  True
s1 = "aabcc", s2 = "dbbca", s3 = "aadbbbaccc"  ->  False
s1 = "",      s2 = "",      s3 = ""            ->  True
```

Ví dụ 1: `"aa" + "dbbc" + "bc" + "a" + "c"` — lấy `aa` từ `s1`, `dbbc` từ `s2`, `bc` từ `s1`, `a` từ `s2`, `c` từ `s1`.

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Quy hoạch động (Dynamic Programming — DP)
2. DP hai chiều (bảng `dp[i][j]`)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Quy hoạch động (Dynamic Programming — DP)

### Khi nào dùng DP?

Quy hoạch động dùng cho bài toán có hai đặc điểm:

1. **Cấu trúc con tối ưu**: đáp án của bài lớn được ghép từ đáp án của các **bài con** nhỏ hơn.
2. **Bài con chồng lấp**: cùng một bài con xuất hiện **nhiều lần** nếu đệ quy thẳng.

Thay vì tính lại, ta **lưu** đáp án mỗi bài con vào bảng và dùng lại → mỗi bài con tính đúng một lần.

### Bốn câu hỏi khi thiết kế DP

1. **Trạng thái**: `dp[i]` nghĩa là gì? (viết thành câu tiếng Việt rõ ràng)
2. **Chuyển trạng thái**: `dp[i]` tính từ những `dp[j]` nào?
3. **Cơ sở**: giá trị nhỏ nhất biết sẵn (`dp[0]`, `dp[1]`…).
4. **Đáp án**: là `dp[n]`, hay `max(dp)`, hay …?

### Ví dụ: leo cầu thang

Mỗi bước leo 1 hoặc 2 bậc. Có bao nhiêu cách leo `n` bậc?

- Trạng thái: `dp[i]` = số cách lên tới bậc `i`.
- Chuyển: bước cuối cùng là 1 bậc (từ `i−1`) hoặc 2 bậc (từ `i−2`) → `dp[i] = dp[i−1] + dp[i−2]`.
- Cơ sở: `dp[0] = 1` (đứng yên), `dp[1] = 1`.

In [1]:
from functools import lru_cache

# Cách 1: đệ quy thuần — chậm vì tính lặp
def leo_de_quy(n):
    return 1 if n <= 1 else leo_de_quy(n - 1) + leo_de_quy(n - 2)

# Cách 2: đệ quy + ghi nhớ (top-down, memoization)
@lru_cache(maxsize=None)
def leo_memo(n):
    return 1 if n <= 1 else leo_memo(n - 1) + leo_memo(n - 2)

# Cách 3: điền bảng từ nhỏ tới lớn (bottom-up, tabulation)
def leo_bang(n):
    dp = [0] * (n + 1)
    dp[0] = 1
    if n >= 1:
        dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n], dp

# Cách 4: chỉ giữ hai giá trị gần nhất — O(1) bộ nhớ
def leo_gon(n):
    a, b = 1, 1                 # dp[i-2], dp[i-1]
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b

print("n = 10:", leo_de_quy(10), leo_memo(10), leo_bang(10)[0], leo_gon(10))
print("bảng dp:", leo_bang(10)[1])

n = 10: 89 89 89 89
bảng dp: [1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


### Top-down hay bottom-up?

| | Top-down (memo) | Bottom-up (bảng) |
|---|---|---|
| Viết | gần với công thức đệ quy, dễ nghĩ | phải chọn **thứ tự** điền bảng |
| Chỉ tính bài con cần thiết | có | không (tính hết) |
| Rủi ro | độ sâu đệ quy | không |
| Tối ưu bộ nhớ | khó | dễ (giữ vài hàng/ô gần nhất) |

### Ví dụ thứ hai: chọn hay không chọn (House Robber)

Dãy nhà có tiền `a[i]`, không được lấy hai nhà **liền nhau**. Lấy tối đa bao nhiêu?

- `dp[i]` = số tiền tối đa khi chỉ xét `i` nhà đầu.
- Nhà thứ `i` (chỉ số `i−1`): **không lấy** → `dp[i−1]`; **lấy** → `dp[i−2] + a[i−1]`. Lấy `max`.

In [2]:
def ke_trom(a):
    dp = [0] * (len(a) + 1)
    if a:
        dp[1] = a[0]
    for i in range(2, len(a) + 1):
        dp[i] = max(dp[i - 1], dp[i - 2] + a[i - 1])
    return dp[-1], dp

print("ke_trom([2, 7, 9, 3, 1]) =", ke_trom([2, 7, 9, 3, 1]))

ke_trom([2, 7, 9, 3, 1]) = (12, [0, 2, 7, 11, 11, 12])


### Lỗi hay gặp

- Định nghĩa trạng thái mơ hồ → công thức sai. Luôn viết `dp[i] = …` bằng lời trước.
- Lệch chỉ số giữa `dp` (dài `n+1`) và mảng đầu vào (dài `n`).
- Điền bảng sai thứ tự (dùng ô chưa tính).
- Quên cơ sở hoặc khởi tạo sai (`0` thay vì `-∞` cho bài tìm max có giá trị âm…).
- Tối ưu bộ nhớ quá sớm khi chưa chắc công thức đúng — hãy viết bản bảng đầy đủ trước.

## 2.2 DP hai chiều (bảng `dp[i][j]`)

### Khi nào cần bảng hai chiều?

Khi trạng thái cần **hai** thông số, thường gặp ở:

- **Lưới**: `dp[i][j]` = đáp án cho ô `(i, j)` (số đường đi, tổng nhỏ nhất…).
- **Hai chuỗi**: `dp[i][j]` = đáp án cho **tiền tố** `s[:i]` và `t[:j]` (edit distance, LCS, xen kẽ…).
- **Đoạn con**: `dp[i][j]` = đáp án cho đoạn `s[i..j]` (palindrome…).

### Ví dụ 1: số đường đi trên lưới

Robot ở góc trên-trái, mỗi bước đi **phải** hoặc **xuống**. Có bao nhiêu đường tới ô `(i, j)`? Bước cuối đến từ ô **trên** hoặc ô **trái**:

```text
dp[i][j] = dp[i-1][j] + dp[i][j-1]
hàng 0 và cột 0 đều = 1 (chỉ một cách: đi thẳng)
```

In [3]:
def so_duong(m, n):
    dp = [[0] * n for _ in range(m)]      # KHÔNG dùng [[0] * n] * m
    for i in range(m):
        for j in range(n):
            if i == 0 or j == 0:
                dp[i][j] = 1
            else:
                dp[i][j] = dp[i - 1][j] + dp[i][j - 1]
    return dp

for row in so_duong(3, 5):
    print(" ".join(f"{v:>3}" for v in row))

  1   1   1   1   1
  1   2   3   4   5
  1   3   6  10  15


Thứ tự điền: theo hàng từ trên xuống, trong hàng từ trái sang → khi tính `dp[i][j]`, ô trên và ô trái **đã có**.

**Tiết kiệm bộ nhớ:** hàng `i` chỉ cần hàng `i−1` → giữ một mảng một chiều `row`, cập nhật `row[j] += row[j-1]` (giá trị cũ của `row[j]` chính là ô trên).

In [4]:
def so_duong_mot_hang(m, n):
    row = [1] * n
    for _ in range(1, m):
        for j in range(1, n):
            row[j] += row[j - 1]          # row[j] (cũ) = trên, row[j-1] (mới) = trái
    return row[-1]
print("so_duong_mot_hang(3, 7) =", so_duong_mot_hang(3, 7), "| so_duong(3, 7) =", so_duong(3, 7)[-1][-1])

so_duong_mot_hang(3, 7) = 28 | so_duong(3, 7) = 28


### Ví dụ 2: hai chuỗi — dãy con chung dài nhất (LCS)

`dp[i][j]` = độ dài dãy con chung dài nhất của `s[:i]` và `t[:j]`. Xét ký tự **cuối** của hai tiền tố:

- `s[i-1] == t[j-1]` → dùng nó: `dp[i][j] = dp[i-1][j-1] + 1`.
- khác → bỏ ký tự cuối của một trong hai: `dp[i][j] = max(dp[i-1][j], dp[i][j-1])`.

Hàng 0 và cột 0 ứng với **chuỗi rỗng** → bằng 0. Đó là lý do bảng có kích thước `(n+1) × (m+1)`.

In [5]:
def lcs(s, t):
    n, m = len(s), len(t)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if s[i - 1] == t[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp

s_demo, t_demo = "abcde", "ace"
bang = lcs(s_demo, t_demo)
print("      ''  " + "   ".join(t_demo))
for i, row in enumerate(bang):
    print(f"{('' if i == 0 else s_demo[i - 1])!r:>4} " + " ".join(f"{v:>3}" for v in row))
print("LCS =", bang[-1][-1])

      ''  a   c   e
  ''   0   0   0   0
 'a'   0   1   1   1
 'b'   0   1   1   1
 'c'   0   1   2   2
 'd'   0   1   2   2
 'e'   0   1   2   3
LCS = 3


### Ví dụ 3: khoảng cách sửa (edit distance) — số phép chèn/xóa/thay ít nhất

`dp[i][j]` = số phép sửa ít nhất biến `s[:i]` thành `t[:j]`.

- Ký tự cuối bằng nhau → `dp[i-1][j-1]`.
- Khác → `1 + min(thay: dp[i-1][j-1], xóa s[i-1]: dp[i-1][j], chèn t[j-1]: dp[i][j-1])`.
- Cơ sở: `dp[i][0] = i` (xóa hết), `dp[0][j] = j` (chèn hết).

In [6]:
def khoang_cach_sua(s, t):
    n, m = len(s), len(t)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1): dp[i][0] = i
    for j in range(m + 1): dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if s[i - 1] == t[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j - 1], dp[i - 1][j], dp[i][j - 1])
    return dp[n][m]

for a_, b_ in [("horse", "ros"), ("kitten", "sitting"), ("", "abc")]:
    print(f"{a_!r} → {b_!r}: {khoang_cach_sua(a_, b_)}")

'horse' → 'ros': 3
'kitten' → 'sitting': 3
'' → 'abc': 3


### Lỗi hay gặp

- `[[0] * n] * m` tạo `m` hàng là **cùng một** list → sửa một ô thì cả cột đổi.
- Lệch chỉ số: bảng `(n+1)×(m+1)` nhưng ký tự là `s[i-1]`, `t[j-1]`.
- Quên hàng/cột cơ sở (chuỗi rỗng, biên lưới).
- Khi tối ưu một hàng: ghi đè giá trị cần dùng lại (ví dụ `dp[i-1][j-1]`) — phải lưu tạm trước.

In [7]:
sai = [[0] * 3] * 2
sai[0][0] = 9
dung = [[0] * 3 for _ in range(2)]
dung[0][0] = 9
print("[[0]*3]*2 sau khi gán ô (0,0):", sai, "| tạo bằng vòng for:", dung)

[[0]*3]*2 sau khi gán ô (0,0): [[9, 0, 0], [9, 0, 0]] | tạo bằng vòng for: [[9, 0, 0], [0, 0, 0]]


## 2.3 Áp dụng vào bài này

**DP hai chuỗi**: `dp[i][j]` = `s1[:i]` và `s2[:j]` có xen kẽ tạo thành `s3[:i+j]` không; ký tự `s3[i+j−1]` phải đến từ `s1[i−1]` (dựa vào `dp[i−1][j]`) hoặc `s2[j−1]` (dựa vào `dp[i][j−1]`). Có thể rút về một hàng.

# 3. Tham lam có đúng không?

Ý tưởng: duyệt `s3`, nếu ký tự khớp `s1` thì lấy từ `s1`, không thì lấy từ `s2`.

In [8]:
from typing import List
from functools import lru_cache
import random

def greedy_interleave(s1: str, s2: str, s3: str) -> bool:
    if len(s1) + len(s2) != len(s3):
        return False
    i = j = 0
    for ch in s3:
        if i < len(s1) and s1[i] == ch:
            i += 1
        elif j < len(s2) and s2[j] == ch:
            j += 1
        else:
            return False
    return True

print(greedy_interleave("aabcc", "dbbca", "aadbbcbcac"), "(đúng: True)")
print(greedy_interleave("ab", "ac", "acab"), "(đúng: True)")

False (đúng: True)
False (đúng: True)


Tham lam sai **cả hai** ví dụ. Xét ví dụ nhỏ `s1 = "ab"`, `s2 = "ac"`, `s3 = "acab"`: ký tự `a` đầu khớp cả hai; tham lam lấy từ `s1`, sau đó cần `c` nhưng `s1` còn `b`, `s2` còn `a` → kẹt. Đúng ra phải lấy `a` từ `s2`. Khi **cả hai** chuỗi khớp, ta không biết chọn bên nào → phải thử cả hai → DP.

# 4. Baseline: đệ quy thử cả hai

`f(i, j)` = có thể đan `s1[i:]` và `s2[j:]` thành `s3[i+j:]` không. Chỉ số trong `s3` luôn là `i + j` — **không cần** tham số thứ ba.

In [9]:
def interleave_rec(s1: str, s2: str, s3: str):
    if len(s1) + len(s2) != len(s3):
        return False, 0
    calls = 0
    def f(i, j):
        nonlocal calls
        calls += 1
        if i == len(s1) and j == len(s2):
            return True
        k = i + j
        if i < len(s1) and s1[i] == s3[k] and f(i + 1, j):
            return True
        if j < len(s2) and s2[j] == s3[k] and f(i, j + 1):
            return True
        return False
    return f(0, 0), calls

print(interleave_rec("aabcc", "dbbca", "aadbbcbcac"))
print(interleave_rec("a" * 10 + "b", "a" * 10, "a" * 20 + "c"))

(True, 12)


(False, 705431)


Chuỗi toàn `a` với ký tự cuối không khớp: thử **mọi** cách trộn — hơn 700 000 lời gọi, dù `s1`, `s2` chỉ dài 11 và 10. Nhưng chỉ có `(m+1)(n+1)` cặp `(i, j)` → memo.

In [10]:
def interleave_memo(s1: str, s2: str, s3: str) -> bool:
    if len(s1) + len(s2) != len(s3):
        return False
    @lru_cache(maxsize=None)
    def f(i, j):
        if i == len(s1) and j == len(s2):
            return True
        k = i + j
        return (i < len(s1) and s1[i] == s3[k] and f(i + 1, j)) or \
               (j < len(s2) and s2[j] == s3[k] and f(i, j + 1))
    return f(0, 0)

print(interleave_memo("a" * 10 + "b", "a" * 10, "a" * 20 + "c"))

False


# 5. Ý tưởng chính: bảng DP — tìm đường trên lưới

`dp[i][j] = True` nếu `s1[:i]` và `s2[:j]` đan được thành `s3[:i+j]`.

Ký tự cuối `s3[i+j-1]` đến từ `s1` hoặc `s2`:

```text
dp[0][0] = True
dp[i][j] = (dp[i-1][j] and s1[i-1] == s3[i+j-1])     # ký tự cuối lấy từ s1
        or (dp[i][j-1] and s2[j-1] == s3[i+j-1])     # ký tự cuối lấy từ s2
```

Hình dung: lưới `(m+1) × (n+1)`, đi **xuống** = lấy một ký tự `s1`, đi **phải** = lấy một ký tự `s2`. Mỗi bước chỉ hợp lệ nếu ký tự lấy ra bằng ký tự tiếp theo của `s3`. Câu hỏi: có đường từ `(0, 0)` tới `(m, n)` không? — giống #62/#63 nhưng các "ô cấm" phụ thuộc vào cạnh đi vào.

In [11]:
def interleave_table(s1: str, s2: str, s3: str) -> bool:
    m, n = len(s1), len(s2)
    if m + n != len(s3):
        return False
    dp = [[False] * (n + 1) for _ in range(m + 1)]
    dp[0][0] = True
    for i in range(m + 1):
        for j in range(n + 1):
            if i > 0 and dp[i - 1][j] and s1[i - 1] == s3[i + j - 1]:
                dp[i][j] = True
            if j > 0 and dp[i][j - 1] and s2[j - 1] == s3[i + j - 1]:
                dp[i][j] = True
    return dp[m][n]

print(interleave_table("aabcc", "dbbca", "aadbbcbcac"), interleave_table("aabcc", "dbbca", "aadbbbaccc"), interleave_table("", "", ""))

True False True


# 6. Mô phỏng: bảng và đường đi

In [12]:
def trace_table(s1: str, s2: str, s3: str):
    m, n = len(s1), len(s2)
    dp = [[False] * (n + 1) for _ in range(m + 1)]
    dp[0][0] = True
    for i in range(m + 1):
        for j in range(n + 1):
            if i > 0 and dp[i - 1][j] and s1[i - 1] == s3[i + j - 1]:
                dp[i][j] = True
            if j > 0 and dp[i][j - 1] and s2[j - 1] == s3[i + j - 1]:
                dp[i][j] = True
    print("s3 =", s3)
    print("        ε " + " ".join(s2) + "   <- s2 (đi phải)")
    for i in range(m + 1):
        label = "ε" if i == 0 else s1[i - 1]
        print(f"     {label}  " + " ".join("T" if x else "." for x in dp[i]))
    print("     ^ s1 (đi xuống)")
    if dp[m][n]:                                     # truy vết một đường
        i, j, moves = m, n, []
        while i or j:
            if i > 0 and dp[i - 1][j] and s1[i - 1] == s3[i + j - 1]:
                moves.append(f"s1:{s1[i-1]}"); i -= 1
            else:
                moves.append(f"s2:{s2[j-1]}"); j -= 1
        print("Một cách đan:", " ".join(reversed(moves)))
    return dp[m][n]

_ = trace_table("aabcc", "dbbca", "aadbbcbcac")

s3 = aadbbcbcac
        ε d b b c a   <- s2 (đi phải)
     ε  T . . . . .
     a  T . . . . .
     a  T T T T T .
     b  . T T . T .
     c  . . T T T T
     c  . . . T . T
     ^ s1 (đi xuống)
Một cách đan: s1:a s1:a s2:d s2:b s2:b s2:c s1:b s1:c s2:a s1:c


Các ô `T` là những trạng thái **đạt được**. Có đường `T` nối `(0, 0)` tới góc dưới phải ⇒ `True`.

# 7. Tối ưu bộ nhớ: một hàng (follow-up)

`dp[i][j]` chỉ cần `dp[i-1][j]` (ô **trên**, chính là `row[j]` trước khi cập nhật) và `dp[i][j-1]` (ô **trái**, `row[j-1]` vừa cập nhật). Một mảng độ dài `n + 1` = `O(len(s2))`.

In [13]:
def interleave_1d(s1: str, s2: str, s3: str) -> bool:
    m, n = len(s1), len(s2)
    if m + n != len(s3):
        return False
    row = [False] * (n + 1)
    for i in range(m + 1):
        for j in range(n + 1):
            if i == 0 and j == 0:
                row[j] = True
            else:
                from_top = i > 0 and row[j] and s1[i - 1] == s3[i + j - 1]
                from_left = j > 0 and row[j - 1] and s2[j - 1] == s3[i + j - 1]
                row[j] = from_top or from_left
    return row[n]

print(interleave_1d("aabcc", "dbbca", "aadbbcbcac"), interleave_1d("aabcc", "dbbca", "aadbbbaccc"))

True False


# 8. Phiên bản đúng format LeetCode

In [14]:
class Solution:
    def isInterleave(self, s1: str, s2: str, s3: str) -> bool:
        m, n = len(s1), len(s2)
        if m + n != len(s3):
            return False
        row = [False] * (n + 1)
        for i in range(m + 1):
            for j in range(n + 1):
                if i == 0 and j == 0:
                    row[j] = True
                else:
                    from_top = i > 0 and row[j] and s1[i - 1] == s3[i + j - 1]
                    from_left = j > 0 and row[j - 1] and s2[j - 1] == s3[i + j - 1]
                    row[j] = from_top or from_left
        return row[n]

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS  {actual}")


def random_interleave(a, b):
    out, i, j = [], 0, 0
    while i < len(a) or j < len(b):
        if j == len(b) or (i < len(a) and random.random() < 0.5):
            out.append(a[i]); i += 1
        else:
            out.append(b[j]); j += 1
    return "".join(out)


sol = Solution()
check("Ví dụ 1", sol.isInterleave("aabcc", "dbbca", "aadbbcbcac"), True)
check("Ví dụ 2", sol.isInterleave("aabcc", "dbbca", "aadbbbaccc"), False)
check("Ví dụ 3", sol.isInterleave("", "", ""), True)
check("Tham lam bị lừa", sol.isInterleave("ab", "ac", "acab"), True)
check("Sai độ dài", sol.isInterleave("a", "b", "abc"), False)
check("Một chuỗi rỗng", sol.isInterleave("", "abc", "abc"), True)

random.seed(97)
for _ in range(4000):
    a = "".join(random.choice("ab") for _ in range(random.randint(0, 6)))
    b = "".join(random.choice("ab") for _ in range(random.randint(0, 6)))
    c = random_interleave(a, b) if random.random() < 0.5 else "".join(random.choice("ab") for _ in range(len(a) + len(b)))
    expected = interleave_rec(a, b, c)[0]
    assert sol.isInterleave(a, b, c) == expected == interleave_table(a, b, c) == interleave_memo(a, b, c), (a, b, c)
print("4000 bộ ngẫu nhiên: khớp đệ quy vét cạn")

Ví dụ 1                PASS  True
Ví dụ 2                PASS  False
Ví dụ 3                PASS  True
Tham lam bị lừa        PASS  True
Sai độ dài             PASS  False
Một chuỗi rỗng         PASS  True


4000 bộ ngẫu nhiên: khớp đệ quy vét cạn


# 9. Vì sao đúng? Độ phức tạp?

Một cách đan `s1[:i]`, `s2[:j]` thành `s3[:i+j]` (với `i + j > 0`) kết thúc bằng ký tự cuối lấy từ `s1` hoặc từ `s2`:

- Từ `s1`: cần `s1[i-1] == s3[i+j-1]` và phần trước là cách đan `s1[:i-1]`, `s2[:j]` → `dp[i-1][j]`.
- Từ `s2`: tương tự → `dp[i][j-1]`.

Ngược lại, mỗi khi một trong hai điều kiện đúng, nối thêm ký tự cuối là được một cách đan. Thứ tự duyệt (hàng trên trước, trái trước) đảm bảo hai ô phụ thuộc đã tính xong.

```text
Time:  O(m · n)
Space: O(n) với một hàng (n = len(s2)); có thể đổi vai để O(min(m, n))
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Tham lam

Như mục 3.

## Lỗi 2: Quên kiểm tra tổng độ dài

In [16]:
def wrong_no_length_check(s1: str, s2: str, s3: str) -> bool:
    m, n = len(s1), len(s2)
    dp = [[False] * (n + 1) for _ in range(m + 1)]
    dp[0][0] = True
    for i in range(m + 1):
        for j in range(n + 1):
            if i + j > len(s3):
                continue
            if i > 0 and dp[i - 1][j] and s1[i - 1] == s3[i + j - 1]:
                dp[i][j] = True
            if j > 0 and dp[i][j - 1] and s2[j - 1] == s3[i + j - 1]:
                dp[i][j] = True
    return dp[m][n]                                # SAI: s3 có thể dài hơn

print("Sai :", wrong_no_length_check("a", "b", "abc"), "| Đúng:", sol.isInterleave("a", "b", "abc"))

Sai : True | Đúng: False


Không kiểm tra thì `s3` dài hơn vẫn có thể trả `True` (còn thừa ký tự không dùng), hoặc ngắn hơn thì `s3[i+j-1]` gây `IndexError`.

## Lỗi 3: Dùng chỉ số `s3[i+j]` thay vì `s3[i+j-1]`

`dp[i][j]` nói về **tiền tố** độ dài `i + j`; ký tự cuối của nó là `s3[i+j-1]`.

## Lỗi 4: Coi "mọi ký tự của `s3` có mặt trong `s1` hoặc `s2`" là đủ

Kiểm tra theo tần suất (`Counter(s1) + Counter(s2) == Counter(s3)`) chỉ là **điều kiện cần** — không quan tâm thứ tự.

In [17]:
from collections import Counter
s1, s2, s3 = "aabcc", "dbbca", "aadbbbaccc"
print("Cùng tần suất?", Counter(s1) + Counter(s2) == Counter(s3), "| Đan được?", sol.isInterleave(s1, s2, s3))

Cùng tần suất? True | Đan được? False


# 11. Mẹo nhận diện pattern

**Hai chuỗi nguồn, một chuỗi đích, giữ thứ tự** → DP `dp[i][j]` trên tiền tố của hai nguồn; chỉ số đích là `i + j`.

Bài liên quan: **#72 Edit Distance**, **#1143 Longest Common Subsequence**, **#115 Distinct Subsequences** (Hard), **#62 Unique Paths** (cùng hình dạng lưới), **#392 Is Subsequence** (một nguồn — hai con trỏ là đủ).

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Lập bảng `dp` cho `s1 = "ab"`, `s2 = "ac"`, `s3 = "acab"`.

## Bài 2 — Điền chỗ trống (bảng 2D)

In [18]:
def exercise_2_is_interleave(s1: str, s2: str, s3: str) -> bool:
    m, n = len(s1), len(s2)
    if m + n != len(s3):
        return False
    dp = [[False] * (n + 1) for _ in range(m + 1)]
    dp[0][0] = True
    for i in range(m + 1):
        for j in range(n + 1):
            # TODO: hai khả năng — từ trên (s1) và từ trái (s2)
            pass
    return dp[m][n]


# assert exercise_2_is_interleave("aabcc", "dbbca", "aadbbcbcac") is True
# assert exercise_2_is_interleave("aabcc", "dbbca", "aadbbbaccc") is False
# assert exercise_2_is_interleave("", "", "") is True

## Bài 3 — Trả về cách đan

Nếu đan được, trả về chuỗi gồm các ký tự `'1'`/`'2'` cho biết mỗi ký tự của `s3` lấy từ chuỗi nào; ngược lại trả `None`.

```python
interleave_labels("ab", "ac", "acab") -> "2211"
```

(Nếu có nhiều cách, trả cách bất kỳ — test sẽ kiểm tra tính hợp lệ.)

In [19]:
def exercise_3_interleave_labels(s1: str, s2: str, s3: str):
    # TODO
    pass


# lab = exercise_3_interleave_labels("ab", "ac", "acab"); assert lab is not None and "".join(c for c, t in zip("acab", lab) if t == "1") == "ab" and "".join(c for c, t in zip("acab", lab) if t == "2") == "ac"
# assert exercise_3_interleave_labels("aabcc", "dbbca", "aadbbbaccc") is None

## Bài 4 — Biến thể: đếm số cách đan

Đếm số **dãy nhãn** `'1'`/`'2'` khác nhau tạo ra `s3` (hai cách khác nhau nếu ở một vị trí nào đó lấy từ chuỗi khác nhau).

```python
count_interleavings("a", "a", "aa") -> 2
count_interleavings("ab", "ac", "acab") -> 1
```

In [20]:
def exercise_4_count_interleavings(s1: str, s2: str, s3: str) -> int:
    # TODO
    pass


# assert exercise_4_count_interleavings("a", "a", "aa") == 2
# assert exercise_4_count_interleavings("ab", "ac", "acab") == 1
# assert exercise_4_count_interleavings("abc", "def", "adbecf") == 1

## Bài 5 — Mini challenge: Is Subsequence (LeetCode #392) và follow-up

`s` có phải dãy con của `t` không (hai con trỏ, `O(len(t))`). Follow-up: có **rất nhiều** chuỗi `s` cần kiểm tra với cùng một `t` dài. Tiền xử lý `t`: với mỗi ký tự, lưu danh sách vị trí; mỗi truy vấn dùng `bisect` để nhảy tới vị trí tiếp theo — `O(len(s) · log len(t))`.

In [21]:
def exercise_5_make_checker(t: str):
    # TODO: trả về hàm is_sub(s) -> bool
    pass


# chk = exercise_5_make_checker("ahbgdc"); assert chk("abc") is True and chk("axc") is False and chk("") is True
# chk = exercise_5_make_checker("aaaa"); assert chk("aaa") is True and chk("aaaaa") is False

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_table` sau khi tự làm.

**Bài 3**
- Mức 1: dựng bảng 2D như Bài 2, rồi truy vết ngược từ `(m, n)` như trong `trace_table`.

**Bài 4**
- Mức 1: thay `or` bằng `+`, `True/False` bằng số: `cnt[i][j] = cnt[i-1][j]·[khớp s1] + cnt[i][j-1]·[khớp s2]`, `cnt[0][0] = 1`.

**Bài 5**
- Mức 1: `pos = defaultdict(list)`; với `k, ch in enumerate(t)`: `pos[ch].append(k)`.
- Mức 2: `cur = -1`; với mỗi ký tự `ch` của `s`: `idx = bisect_right(pos[ch], cur)`; nếu hết thì False; `cur = pos[ch][idx]`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [22]:
_ = trace_table("ab", "ac", "acab")

s3 = acab
        ε a c   <- s2 (đi phải)
     ε  T T T
     a  T . T
     b  . . T
     ^ s1 (đi xuống)
Một cách đan: s2:a s2:c s1:a s1:b


## Đáp án Bài 2

In [23]:
def answer_2_is_interleave(s1: str, s2: str, s3: str) -> bool:
    return interleave_table(s1, s2, s3)

assert answer_2_is_interleave("aabcc", "dbbca", "aadbbcbcac") is True
assert answer_2_is_interleave("aabcc", "dbbca", "aadbbbaccc") is False
assert answer_2_is_interleave("", "", "") is True
print("Bài 2 PASS (giống interleave_table ở mục 5)")

Bài 2 PASS (giống interleave_table ở mục 5)


## Đáp án Bài 3

In [24]:
def answer_3_interleave_labels(s1: str, s2: str, s3: str):
    m, n = len(s1), len(s2)
    if m + n != len(s3):
        return None
    dp = [[False] * (n + 1) for _ in range(m + 1)]
    dp[0][0] = True
    for i in range(m + 1):
        for j in range(n + 1):
            if i > 0 and dp[i - 1][j] and s1[i - 1] == s3[i + j - 1]:
                dp[i][j] = True
            if j > 0 and dp[i][j - 1] and s2[j - 1] == s3[i + j - 1]:
                dp[i][j] = True
    if not dp[m][n]:
        return None
    labels, i, j = [], m, n
    while i or j:
        if i > 0 and dp[i - 1][j] and s1[i - 1] == s3[i + j - 1]:
            labels.append("1"); i -= 1
        else:
            labels.append("2"); j -= 1
    return "".join(reversed(labels))

lab = answer_3_interleave_labels("ab", "ac", "acab"); assert lab is not None and "".join(c for c, t in zip("acab", lab) if t == "1") == "ab" and "".join(c for c, t in zip("acab", lab) if t == "2") == "ac"
assert answer_3_interleave_labels("aabcc", "dbbca", "aadbbbaccc") is None
print("Bài 3 PASS |", answer_3_interleave_labels("aabcc", "dbbca", "aadbbcbcac"))

Bài 3 PASS | 1122221121


## Đáp án Bài 4

In [25]:
def answer_4_count_interleavings(s1: str, s2: str, s3: str) -> int:
    m, n = len(s1), len(s2)
    if m + n != len(s3):
        return 0
    cnt = [[0] * (n + 1) for _ in range(m + 1)]
    cnt[0][0] = 1
    for i in range(m + 1):
        for j in range(n + 1):
            if i > 0 and s1[i - 1] == s3[i + j - 1]:
                cnt[i][j] += cnt[i - 1][j]
            if j > 0 and s2[j - 1] == s3[i + j - 1]:
                cnt[i][j] += cnt[i][j - 1]
    return cnt[m][n]

assert answer_4_count_interleavings("a", "a", "aa") == 2
assert answer_4_count_interleavings("ab", "ac", "acab") == 1
assert answer_4_count_interleavings("abc", "def", "adbecf") == 1
for _ in range(2000):
    a = "".join(random.choice("ab") for _ in range(random.randint(0, 5)))
    b = "".join(random.choice("ab") for _ in range(random.randint(0, 5)))
    c = random_interleave(a, b)
    assert (answer_4_count_interleavings(a, b, c) > 0) == sol.isInterleave(a, b, c)
print("Bài 4 PASS | số cách đan 'aaa' + 'aaa' thành 'a'*6:", answer_4_count_interleavings("aaa", "aaa", "a" * 6), "= C(6, 3)")

Bài 4 PASS | số cách đan 'aaa' + 'aaa' thành 'a'*6: 20 = C(6, 3)


## Đáp án Bài 5

In [26]:
from bisect import bisect_right
from collections import defaultdict

def answer_5_make_checker(t: str):
    pos = defaultdict(list)
    for k, ch in enumerate(t):
        pos[ch].append(k)
    def is_sub(s: str) -> bool:
        cur = -1
        for ch in s:
            lst = pos.get(ch, [])
            idx = bisect_right(lst, cur)
            if idx == len(lst):
                return False
            cur = lst[idx]
        return True
    return is_sub

chk = answer_5_make_checker("ahbgdc"); assert chk("abc") is True and chk("axc") is False and chk("") is True
chk = answer_5_make_checker("aaaa"); assert chk("aaa") is True and chk("aaaaa") is False

def two_pointers(s, t):
    i = 0
    for ch in t:
        if i < len(s) and s[i] == ch:
            i += 1
    return i == len(s)

for _ in range(2000):
    t = "".join(random.choice("abc") for _ in range(random.randint(0, 10)))
    s = "".join(random.choice("abc") for _ in range(random.randint(0, 5)))
    assert answer_5_make_checker(t)(s) == two_pointers(s, t)
print("Bài 5 PASS")

Bài 5 PASS


# 15. Tổng kết

> `dp[i][j]`: `s1[:i]` và `s2[:j]` đan được thành `s3[:i+j]`. `dp[i][j] = (dp[i-1][j] ∧ s1[i-1] = s3[i+j-1]) ∨ (dp[i][j-1] ∧ s2[j-1] = s3[i+j-1])`. Kiểm tra tổng độ dài trước. Một hàng đủ cho follow-up.

---

## Checklist trước khi nộp

- [ ] Kiểm tra `len(s1) + len(s2) == len(s3)`?
- [ ] Ký tự đích là `s3[i + j - 1]`?
- [ ] `dp[0][0] = True` và xử lý hàng 0 / cột 0?
- [ ] Không dùng tham lam?